# 多城市 LOD2 屋顶检索候选库

本 notebook 将 Luxembourg 已冻结的同源 G1/G2 指标流程扩展到数据库中全部 19 个 LOD2 城市。每个城市按固定随机顺序分批处理，硬筛后最多保留 5,000 个街区，再建立跨城市候选库并重新检索 Luxembourg 的正式查询。

这一步不使用 Isolation Forest，也不重新选择指标或调整主权重。完整运行会读取大量 LOD2 表面，建议先将 `CITIES_TO_RUN` 设为 `['luxembourg']` 检查环境，再恢复为 `None` 运行全部城市。

In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata as metadata
import json
import os
from datetime import datetime, timezone
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_notebook_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'multicity_database.py').is_file():
        return current
    for parent in (current, *current.parents):
        candidate = parent / 'casebase' / 'lod2_roof_retrieval_experiment'
        if (candidate / 'multicity_database.py').is_file():
            return candidate
    raise FileNotFoundError('请从 block-gml-casebase 项目内运行本 notebook。')

NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import block_metrics as bm
import multicity_database as mdb
import multicity_pipeline as mp
import retrieval as rt

for module in (bm, mdb, mp, rt):
    importlib.reload(module)


## 1. 固定配置

主配置来自 Luxembourg 开发阶段和多城市扩容设计。`CITIES_TO_RUN=None` 表示运行全部目录城市；如果中途停止，保持配置不变再次运行会复用城市检查点。

In [ ]:
BASE_SEED = 20260909
EXPERIMENT_SCHEMA = 'exp_lod2_roof_multi'
EXPECTED_CITY_COUNT = 19
TARGET_PER_CITY = 5000
BATCH_SIZE = 1000
MIN_REFERENCE_BCR = 0.02
COVERAGE_THRESHOLD = 0.80
POOL_SIZE = 50
TOP_K = 5
PRIMARY_LAMBDA = 0.35
WRITE_EXPERIMENT_SCHEMA = True
RESUME_FROM_CHECKPOINTS = True
CITIES_TO_RUN = None  # 例如 ['luxembourg']；None 表示全部 19 城市
ENV_PATH = None  # 找不到项目 .env 时填写绝对路径

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'multicity'
CITY_OUTPUT_DIR = OUTPUT_DIR / 'cities'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_CONFIG = {
    'base_seed': BASE_SEED,
    'target_per_city': TARGET_PER_CITY,
    'batch_size': BATCH_SIZE,
    'min_reference_bcr': MIN_REFERENCE_BCR,
    'coverage_threshold': COVERAGE_THRESHOLD,
    'm1_features': mp.FROZEN_M1_FEATURES,
    'roof_features': bm.ROOF_FEATURES,
    'pool_size': POOL_SIZE,
    'top_k': TOP_K,
    'primary_lambda': PRIMARY_LAMBDA,
}
CONFIG_FINGERPRINT = mp.config_fingerprint(RUN_CONFIG)
pd.Series({'config_fingerprint': CONFIG_FINGERPRINT, **RUN_CONFIG})


## 2. 读取并核验城市目录

这里只执行只读查询。每个城市必须具有标准街区、建筑、表面字段和明确的 `lod2_metric_srid`；不合格城市会记录失败原因，其他城市仍可继续。

In [ ]:
connection = mdb.connect_from_env(ENV_PATH)
catalog_sources = mdb.discover_lod2_cities(connection)
if CITIES_TO_RUN is None and len(catalog_sources) != EXPECTED_CITY_COUNT:
    raise RuntimeError(
        f'数据库目录发现 {len(catalog_sources)} 个 LOD2 城市，预期 {EXPECTED_CITY_COUNT} 个。'
    )
requested = None if CITIES_TO_RUN is None else {str(city).lower() for city in CITIES_TO_RUN}
sources = [source for source in catalog_sources if requested is None or source.city in requested]
if requested is not None:
    missing_requested = sorted(requested - {source.city for source in sources})
    if missing_requested:
        raise ValueError('未在 LOD2 城市目录中找到：' + ', '.join(missing_requested))

candidate_ids_by_city = {}
valid_sources = []
inventory_rows = []
for source in sources:
    try:
        mdb.validate_source_columns(connection, source)
        candidate_ids = mdb.fetch_candidate_block_ids(connection, source)
        candidate_ids_by_city[source.city] = candidate_ids
        valid_sources.append(source)
        inventory_rows.append({
            'city': source.city, 'display_name': source.display_name,
            'metric_srid': source.metric_srid, 'candidate_blocks': len(candidate_ids),
            'city_seed': mp.stable_city_seed(BASE_SEED, source.city), 'status': 'ready',
        })
    except Exception as error:
        connection.rollback()
        inventory_rows.append({
            'city': source.city, 'display_name': source.display_name,
            'metric_srid': source.metric_srid, 'candidate_blocks': np.nan,
            'city_seed': mp.stable_city_seed(BASE_SEED, source.city),
            'status': f'failed: {type(error).__name__}: {error}',
        })

city_inventory = pd.DataFrame(inventory_rows)
if not valid_sources:
    raise RuntimeError('没有通过字段核验的 LOD2 城市。')
if WRITE_EXPERIMENT_SCHEMA:
    assert EXPERIMENT_SCHEMA == mdb.EXPERIMENT_SCHEMA
    for relation in (mdb.SAMPLE_TABLE, mdb.BUILDING_TABLE, mdb.BLOCK_TABLE, mdb.RETRIEVAL_TABLE):
        mdb.assert_multicity_relation(relation)
    mdb.ensure_multicity_schema(connection)
city_inventory


## 3. 分批计算与硬筛

数据条件排除不可计算或明显不合理的对象；`BCR < 0.02` 单独记为研究范围排除。候选街区先按城市固定种子随机排列，再取最先通过硬筛的 5,000 个，因此结果不依赖数据库返回顺序。

In [ ]:
def measure_city_batch(connection, source, block_ids):
    blocks = mdb.fetch_city_blocks(connection, source, block_ids)
    records = mdb.iter_city_buildings(connection, source, block_ids)
    buildings = bm.compute_building_metrics(records)
    block_frame = bm.compute_block_metrics(
        blocks, buildings, coverage_threshold=COVERAGE_THRESHOLD
    )
    return block_frame, buildings


## 4. 逐城市建立最终样本

每个城市完成后立即保存四个 parquet 和一个配置元数据文件。配置指纹相同且文件齐全时可直接恢复；改变筛选阈值、指标或抽样参数后会自动拒绝旧检查点。

In [ ]:
city_summary_rows = []
failed_city_rows = []

def city_checkpoint_paths(city):
    city_dir = CITY_OUTPUT_DIR / city
    return {
        'manifest': city_dir / 'sample_manifest.parquet',
        'processed_blocks': city_dir / 'processed_block_metrics.parquet',
        'selected_blocks': city_dir / 'selected_block_metrics.parquet',
        'selected_buildings': city_dir / 'selected_building_metrics.parquet',
        'metadata': city_dir / 'checkpoint_config.json',
    }

for city_index, source in enumerate(valid_sources, start=1):
    paths = city_checkpoint_paths(source.city)
    city_fingerprint = mp.config_fingerprint({
        **RUN_CONFIG, 'city': source.city, 'metric_srid': source.metric_srid,
        'candidate_ids_fingerprint': mp.id_set_fingerprint(
            candidate_ids_by_city[source.city]
        ),
    })
    try:
        checkpoint = (
            mp.load_city_checkpoint(paths, city_fingerprint)
            if RESUME_FROM_CHECKPOINTS else None
        )
        if checkpoint is not None:
            manifest = checkpoint['manifest']
            processed_blocks = checkpoint['processed_blocks']
            selected_blocks = checkpoint['selected_blocks']
            selected_buildings = checkpoint['selected_buildings']
            mode = 'checkpoint'
        else:
            ordered_ids = mp.randomized_block_ids(
                candidate_ids_by_city[source.city], BASE_SEED, source.city
            )
            result = mp.collect_city_sample(
                ordered_ids=ordered_ids,
                measure_batch=lambda ids, source=source: measure_city_batch(connection, source, ids),
                target_count=TARGET_PER_CITY, batch_size=BATCH_SIZE,
                min_bcr=MIN_REFERENCE_BCR,
            )
            manifest = result.manifest.copy()
            processed_blocks = result.processed_blocks.copy()
            selected_blocks = result.selected_blocks.copy()
            selected_buildings = result.selected_buildings.copy()
            for frame in (manifest, processed_blocks, selected_blocks, selected_buildings):
                frame.insert(0, 'city', source.city)
            mp.write_city_checkpoint(
                paths,
                {
                    'manifest': manifest,
                    'processed_blocks': processed_blocks,
                    'selected_blocks': selected_blocks,
                    'selected_buildings': selected_buildings,
                },
                {'fingerprint': city_fingerprint, 'city': source.city,
                 'metric_srid': source.metric_srid,
                 'candidate_ids_fingerprint': mp.id_set_fingerprint(
                     candidate_ids_by_city[source.city]
                 )},
            )
            mode = 'computed'

        if WRITE_EXPERIMENT_SCHEMA:
            mdb.replace_city_metrics(
                connection, source.city, manifest, selected_buildings, processed_blocks
            )

        processed_mask = manifest['processed'].fillna(False).astype(bool)
        city_summary_rows.append({
            'city': source.city, 'mode': mode,
            'candidate_blocks': len(manifest),
            'processed_blocks': int(processed_mask.sum()),
            'data_valid_blocks': int(manifest['data_valid'].fillna(False).sum()),
            'eligible_blocks': int(manifest['is_eligible'].fillna(False).sum()),
            'selected_blocks': int(manifest['selected'].sum()),
            'low_bcr_excluded': int(
                manifest['exclusion_codes'].fillna('').str.contains('below_reference_bcr').sum()
            ),
        })
        print(
            f'[{city_index:02d}/{len(valid_sources):02d}] {source.city}: '
            f"{len(selected_blocks):,} selected ({mode})"
        )
        del manifest, processed_blocks, selected_blocks, selected_buildings
    except Exception as error:
        connection.rollback()
        failed_city_rows.append({
            'city': source.city, 'error_type': type(error).__name__, 'message': str(error)
        })
        print(f'[{city_index:02d}/{len(valid_sources):02d}] {source.city}: FAILED - {error}')


## 5. 合并候选库并检查城市构成

这里只合并街区级指标；建筑级指标保留在各城市检查点和实验 schema 中。单城补跑后会重新读取所有兼容检查点，只有19城齐全时才生成或覆盖正式跨城市检索结果。

In [ ]:
city_summary = pd.DataFrame(city_summary_rows).sort_values('city').reset_index(drop=True)
failed_cities = pd.DataFrame(failed_city_rows)
city_block_frames = []
loaded_pool_cities = []
for source in catalog_sources:
    if source.city not in candidate_ids_by_city:
        continue
    fingerprint = mp.config_fingerprint({
        **RUN_CONFIG, 'city': source.city, 'metric_srid': source.metric_srid,
        'candidate_ids_fingerprint': mp.id_set_fingerprint(
            candidate_ids_by_city[source.city]
        ),
    })
    checkpoint = mp.load_city_checkpoint(
        city_checkpoint_paths(source.city), fingerprint
    )
    if checkpoint is not None:
        city_block_frames.append(checkpoint['selected_blocks'])
        loaded_pool_cities.append(source.city)
if not city_block_frames:
    raise RuntimeError('没有城市形成可用的最终街区样本。')
multicity_blocks = pd.concat(city_block_frames, ignore_index=True)
GLOBAL_POOL_COMPLETE = (
    CITIES_TO_RUN is None
    and not failed_city_rows
    and len(valid_sources) == len(catalog_sources)
    and len(city_summary_rows) == len(catalog_sources)
    and set(loaded_pool_cities) == {source.city for source in catalog_sources}
)
partial_id = mp.config_fingerprint({
    'config': CONFIG_FINGERPRINT, 'cities': sorted(loaded_pool_cities)
})
RESULT_OUTPUT_DIR = (
    OUTPUT_DIR if GLOBAL_POOL_COMPLETE
    else OUTPUT_DIR / 'partial' / partial_id
)
RESULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if multicity_blocks[['city', 'block_id']].astype(str).duplicated().any():
    duplicates = multicity_blocks.loc[
        multicity_blocks[['city', 'block_id']].astype(str).duplicated(False),
        ['city', 'block_id'],
    ].astype(str).agg(':'.join, axis=1).unique()
    raise RuntimeError('城市内 block_id 不唯一：' + ', '.join(duplicates[:10]))

display(city_summary)
if not failed_cities.empty:
    display(failed_cities)
pd.Series({
    'loaded_pool_cities': int(multicity_blocks['city'].nunique()),
    'global_pool_complete': GLOBAL_POOL_COMPLETE,
    'final_blocks': len(multicity_blocks),
    'min_city_blocks': int(multicity_blocks.groupby('city').size().min()),
    'max_city_blocks': int(multicity_blocks.groupby('city').size().max()),
})


## 6. 多城市指标诊断

相关性和逐城市中位数只用于发现来源偏移，不据此重新删除指标。M1始终使用冻结的11项基线。

In [ ]:
analysis_features = mp.FROZEN_M1_FEATURES + list(bm.ROOF_FEATURES)
city_feature_medians = multicity_blocks.groupby('city')[analysis_features].median()
feature_correlations = multicity_blocks[analysis_features].corr(method='spearman')
display(city_feature_medians)
display(feature_correlations.round(2))


## 7. 跨城市重新检索 Luxembourg 正式查询

所有城市共同拟合稳健缩放。M1从全部候选中按总体形态距离形成最近50个的共享候选池；M2只在该池中用固定权重 `0.65 × D_general + 0.35 × D_roof` 重排。

In [ ]:
formal_queries = mdb.fetch_luxembourg_formal_queries(connection)
luxembourg_ids = set(multicity_blocks.loc[
    multicity_blocks['city'].astype(str).eq('luxembourg'), 'block_id'
].astype(str))
eligible_queries = formal_queries.loc[
    formal_queries['block_id'].astype(str).isin(luxembourg_ids)
].copy()
ineligible_queries = formal_queries.loc[
    ~formal_queries['block_id'].astype(str).isin(luxembourg_ids)
].copy()
general_scaler = None
roof_scaler = None
cross_city_results = pd.DataFrame()
retrieval_summary = pd.DataFrame()
if GLOBAL_POOL_COMPLETE:
    if eligible_queries.empty:
        raise RuntimeError('现有 Luxembourg 正式查询均未进入多城市最终样本。')
    general_scaler = rt.RobustFeatureScaler.fit(
        multicity_blocks, mp.FROZEN_M1_FEATURES
    )
    roof_scaler = rt.RobustFeatureScaler.fit(multicity_blocks, bm.ROOF_FEATURES)
    query_metadata = eligible_queries.set_index('block_id')[['query_role', 'stratum']]
    retrieval_parts = []
    for query_id in eligible_queries['block_id'].astype(str):
        result = rt.retrieve_for_query(
            multicity_blocks, query_id, query_city='luxembourg',
            general_features=mp.FROZEN_M1_FEATURES, roof_features=bm.ROOF_FEATURES,
            pool_size=POOL_SIZE, top_k=TOP_K, lambda_values=[PRIMARY_LAMBDA],
            general_scaler=general_scaler, roof_scaler=roof_scaler,
        )
        result['query_role'] = query_metadata.loc[query_id, 'query_role']
        result['stratum'] = query_metadata.loc[query_id, 'stratum']
        retrieval_parts.append(result)

    cross_city_results = pd.concat(retrieval_parts, ignore_index=True)
    cross_city_results['candidate_city'] = cross_city_results['candidate_city'].astype(str)
    retrieval_summary = (
        cross_city_results.groupby('method', as_index=False)
        .agg(
            mean_general_distance=('d_general', 'mean'),
            mean_roof_distance=('d_roof', 'mean'),
            cross_city_share=('same_city', lambda values: float((~values).mean())),
            represented_candidate_cities=('candidate_city', 'nunique'),
        )
    )
else:
    print(
        f'当前只有 {len(loaded_pool_cities)}/{len(catalog_sources)} 个兼容城市检查点；'
        '已安全跳过正式跨城市检索，不会覆盖完整结果。'
    )
display(pd.Series({
    'formal_queries': len(formal_queries),
    'eligible_queries': len(eligible_queries),
    'ineligible_queries': len(ineligible_queries),
}))
display(retrieval_summary)
display(cross_city_results.head(20))


## 8. 保存合并结果与运行配置

保存文件不包含几何大对象。后续制图可以根据结果中的城市和 `block_id` 回到对应源表读取少量几何。

In [ ]:
multicity_blocks.to_parquet(RESULT_OUTPUT_DIR / 'multicity_block_metrics.parquet', index=False)
city_inventory.to_csv(RESULT_OUTPUT_DIR / 'city_inventory.csv', index=False, encoding='utf-8-sig')
city_summary.to_csv(RESULT_OUTPUT_DIR / 'city_summary.csv', index=False, encoding='utf-8-sig')
failed_cities.to_csv(RESULT_OUTPUT_DIR / 'failed_cities.csv', index=False, encoding='utf-8-sig')
city_feature_medians.to_csv(RESULT_OUTPUT_DIR / 'city_feature_medians.csv', encoding='utf-8-sig')
feature_correlations.to_csv(RESULT_OUTPUT_DIR / 'feature_correlations.csv', encoding='utf-8-sig')
eligible_queries.to_csv(RESULT_OUTPUT_DIR / 'eligible_luxembourg_queries.csv', index=False, encoding='utf-8-sig')
ineligible_queries.to_csv(RESULT_OUTPUT_DIR / 'ineligible_luxembourg_queries.csv', index=False, encoding='utf-8-sig')
if GLOBAL_POOL_COMPLETE:
    cross_city_results.to_parquet(RESULT_OUTPUT_DIR / 'cross_city_retrieval_results.parquet', index=False)
    cross_city_results.to_csv(RESULT_OUTPUT_DIR / 'cross_city_retrieval_results.csv', index=False, encoding='utf-8-sig')
    retrieval_summary.to_csv(RESULT_OUTPUT_DIR / 'retrieval_summary.csv', index=False, encoding='utf-8-sig')

if WRITE_EXPERIMENT_SCHEMA and GLOBAL_POOL_COMPLETE:
    mdb.replace_city_retrieval_results(connection, 'luxembourg', cross_city_results)

experiment_config = {
    **RUN_CONFIG,
    'execution_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'config_fingerprint': CONFIG_FINGERPRINT,
    'experiment_schema': EXPERIMENT_SCHEMA,
    'catalog_city_count': len(catalog_sources),
    'loaded_pool_cities': sorted(loaded_pool_cities),
    'global_pool_complete': GLOBAL_POOL_COMPLETE,
    'final_block_count': len(multicity_blocks),
    'general_scaler_medians': general_scaler.medians.to_dict() if general_scaler else None,
    'general_scaler_iqrs': general_scaler.iqrs.to_dict() if general_scaler else None,
    'roof_scaler_medians': roof_scaler.medians.to_dict() if roof_scaler else None,
    'roof_scaler_iqrs': roof_scaler.iqrs.to_dict() if roof_scaler else None,
    'package_versions': {name: metadata.version(name) for name in [
        'numpy', 'pandas', 'scipy', 'scikit-learn', 'shapely', 'psycopg2', 'pyarrow'
    ]},
}
with (RESULT_OUTPUT_DIR / 'experiment_config.json').open('w', encoding='utf-8') as file:
    json.dump(experiment_config, file, ensure_ascii=False, indent=2)
connection.close()
print(f'已保存 {len(multicity_blocks):,} 个街区和 {len(cross_city_results):,} 条检索结果到 {RESULT_OUTPUT_DIR}')


## 9. 运行后首先查看什么

1. `city_summary`：各城市处理量、有效量和低 BCR 排除量；
2. `failed_cities`：是否存在表结构或几何处理失败；
3. `retrieval_summary`：M1/M2 的总体距离、屋顶距离和跨城市候选比例；
4. `cross_city_retrieval_results.csv`：下一步可视化所需的查询、候选城市和排名。

本 notebook 不根据运行结果自动修改阈值、指标或权重。